# AdventureWorks on Databricks

Predict over the AdventureWorks sample where its tables already sit, in a Unity Catalog schema.
Nothing is copied out: the client samples the tables with SQL, a hop at a time, and only the rows a prediction needs leave the catalog.

You need the four AdventureWorks tables (`adventureworks_customers`, `adventureworks_products`, `adventureworks_sales_order_headers`, `adventureworks_sales_order_details`) in one schema, and a Kumo Relational model on a Model Serving endpoint.

Sampling runs through this notebook's own Spark session on Databricks Runtime 17.3 LTS or later, or serverless environment 4 or later.
On an older runtime, from 14.3 LTS, name a SQL warehouse instead.

If the schema also holds a metric view over these tables, name it and two of the questions are asked through the view as well.
A metric view is Databricks' form of a semantic model: tables, how they join, and their dimensions.

In [ ]:
dbutils.widgets.text(
    'install',
    'sdm-client[relational,databricks,databricks-serving]',
    'pip install',
)
dbutils.widgets.text('catalog', 'main', 'Catalog')
dbutils.widgets.text('schema', 'adventureworks', 'Schema')
dbutils.widgets.text('endpoint', '', 'Model Serving endpoint')
dbutils.widgets.text('warehouse_id', '', 'SQL warehouse ID, or empty for Spark')
dbutils.widgets.text('metric_view', '', 'Metric view, or empty to skip')

install = dbutils.widgets.get('install')

# The runtime's packages are built against the NumPy it ships; pinning it keeps
# the install from upgrading NumPy beneath them, which stops Python starting.
import numpy

keep_numpy = f'numpy=={numpy.__version__}'

In [ ]:
%pip install --only-binary=:all: $install $keep_numpy

In [ ]:
dbutils.library.restartPython()

In [ ]:
import json
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version


def installed(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None


ENVIRONMENT = {
    'runtime': os.environ.get('DATABRICKS_RUNTIME_VERSION'),
    'python': sys.version.split()[0],
    'machine': platform.machine(),
    **{
        package: installed(package)
        for package in (
            'sdm-client',
            'sdm-relational-engine',
            'numpy',
            'pandas',
            'pyarrow',
            'pydantic',
            'databricks-sdk',
            'databricks-sql-connector',
        )
    },
}
print(json.dumps(ENVIRONMENT, indent=1))

## 1. Say where the tables are

A `DatabricksSource` is a connection and the catalog and schema its tables live in.
With no warehouse named, it is this notebook's `spark`: the notebook's own compute and identity, and nothing else to set up.
With one named, `warehouse()` opens a connection to it as the notebook's own `WorkspaceClient`.

In [ ]:
from databricks.sdk import WorkspaceClient
from sdm_connectors import DatabricksSource
from sdm_connectors.databricks import warehouse

catalog = dbutils.widgets.get('catalog')
schema_name = dbutils.widgets.get('schema')
endpoint = dbutils.widgets.get('endpoint')
warehouse_id = dbutils.widgets.get('warehouse_id')
if not endpoint:
    raise ValueError(
        'Set the endpoint widget to a Kumo Relational serving endpoint.'
    )

workspace = WorkspaceClient()
if warehouse_id:
    connection = warehouse(warehouse_id, workspace=workspace)
else:
    connection = spark

source = DatabricksSource(connection, catalog=catalog, schema=schema_name)
source

## 2. Describe the tables

`Schema.from_source` reads column types from the catalog and semantic types from a preview; no table is read whole.
Each table gets the short name the queries use.

These tables declare no keys in the catalog, so the keys and links are stated here.
The order headers are left without one: nothing below links to them through it.

In [ ]:
from sdm_client import Relationship, Schema

schema = Schema.from_source(
    source,
    {
        'customers': 'adventureworks_customers',
        'products': 'adventureworks_products',
        'sales_order_headers': 'adventureworks_sales_order_headers',
        'sales_order_details': 'adventureworks_sales_order_details',
    },
    primary_keys={
        'customers': 'CustomerID',
        'products': 'ProductID',
        'sales_order_details': 'SalesOrderDetailID',
    },
    time_columns={
        'sales_order_headers': 'OrderDate',
        'sales_order_details': 'OrderDate',
    },
    relationships=[
        Relationship(
            'sales_order_headers', 'CustomerID', 'customers', 'CustomerID'
        ),
        Relationship(
            'sales_order_details', 'CustomerID', 'customers', 'CustomerID'
        ),
        Relationship(
            'sales_order_details', 'ProductID', 'products', 'ProductID'
        ),
    ],
)

A date on a lookup table records when its row was written, not an event, so customers and products get no time column.

`rowguid` is a per-row UUID AdventureWorks carries for replication.
It identifies a row and says nothing about it; left in on the detail table, it drives the demand forecast to exactly zero.
Leaving a column out means it is neither read nor sent.

In [ ]:
schema['customers'].time_column = None
schema['products'].time_column = None
for table in schema.tables.values():
    if 'rowguid' in table:
        del table['rowguid']

for table in schema.tables.values():
    print(table)
print(schema.relationships)

## 3. Ask

A 30-day demand forecast for the best-selling product, now and as of six months before the data ends; the same forecast three windows out; which frequent customers go quiet for 90 days; which of the most widely bought products one customer buys next; and a product's colour.

In [ ]:
import pandas as pd


def table(name):
    return f'`{catalog}`.`{schema_name}`.adventureworks_{name}'


def rows(query):
    return spark.sql(query).collect()


top_product = rows(
    f'SELECT ProductID FROM {table("sales_order_details")} '
    'GROUP BY ProductID ORDER BY sum(OrderQty) DESC, ProductID LIMIT 1'
)[0][0]
# Bought by the most customers, so each has buyers in the context: a ranking
# is asked as one yes-or-no question per candidate, and a candidate nobody in
# the context bought has only one answer to learn from.
popular_products = [
    row[0]
    for row in rows(
        f'SELECT ProductID FROM {table("sales_order_details")} '
        'GROUP BY ProductID ORDER BY count(DISTINCT CustomerID) DESC, ProductID LIMIT 10'
    )
]
frequent_customers = [
    row[0]
    for row in rows(
        f'SELECT CustomerID FROM {table("sales_order_headers")} '
        'GROUP BY CustomerID ORDER BY count(*) DESC, CustomerID LIMIT 25'
    )
]
latest = rows(f'SELECT max(OrderDate) FROM {table("sales_order_details")}')[0][
    0
]
anchor = pd.Timestamp(latest) - pd.Timedelta(days=180)
print(
    f'top product {top_product}; latest order {latest}; anchor {anchor.date()}'
)

In [ ]:
# Four estimators over shuffled columns: one estimator's answer moves with
# column order, and averaging over four permutations settles it.
ENSEMBLE = {'num_estimators': 4, 'column_shuffle': True}
DEMAND = (
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FOR EACH products.ProductID'
)

QUESTIONS = {
    'demand_30d': (
        DEMAND,
        {'indices': [top_product], 'inference_config': ENSEMBLE},
    ),
    'demand_30d_historical': (
        DEMAND,
        {
            'indices': [top_product],
            'anchor_time': anchor,
            'inference_config': ENSEMBLE,
        },
    ),
    'demand_3_timeframes': (
        'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
        'FORECAST 3 TIMEFRAMES FOR EACH products.ProductID',
        {
            'indices': [top_product],
            'anchor_time': anchor,
            'inference_config': ENSEMBLE,
        },
    ),
    'inactivity_90d': (
        'PREDICT COUNT(sales_order_headers.*, 0, 90, days)=0 '
        'FOR EACH customers.CustomerID',
        {'indices': frequent_customers},
    ),
    'recommend_top5': (
        'PREDICT LIST_DISTINCT(sales_order_details.ProductID, 0, 30, days) '
        'RANK TOP 5 FOR EACH customers.CustomerID',
        {
            'indices': frequent_customers[:1],
            'candidates': popular_products,
            'anchor_time': anchor,
            'num_context_samples': 1000,
        },
    ),
    'product_color': (
        'PREDICT products.Color FOR EACH products.ProductID',
        {'indices': [top_product]},
    ),
}

### Through a metric view

A metric view already says which tables a question is about and how they join, so it can stand in for the tables named above.
The source hands it out as a semantic model, which is a source of its own: `source.semantic_model(name)`.
Its source table keeps its catalog name, each join becomes a table named after the join, and each dimension becomes a column, computed in the warehouse where it is an expression.
Measures are aggregations, so the question states its own.

This section runs when the `metric_view` widget names one, for example a view created with:

```sql
CREATE VIEW main.adventureworks.adventureworks_sales_metrics
WITH METRICS LANGUAGE YAML AS $$
version: 0.1
source: main.adventureworks.adventureworks_sales_order_details
joins:
  - name: customers
    source: main.adventureworks.adventureworks_customers
    on: source.CustomerID = customers.CustomerID
  - name: products
    source: main.adventureworks.adventureworks_products
    on: source.ProductID = products.ProductID
dimensions:
  - name: SalesOrderDetailID
    expr: source.SalesOrderDetailID
  - name: SalesOrderID
    expr: source.SalesOrderID
  - name: OrderDate
    expr: source.OrderDate
  - name: OrderQty
    expr: source.OrderQty
  - name: UnitPrice
    expr: source.UnitPrice
  - name: LineTotal
    expr: source.LineTotal
  - name: Discounted
    expr: source.UnitPriceDiscount > 0
  - name: TerritoryID
    expr: customers.TerritoryID
  - name: StoreID
    expr: customers.StoreID
  - name: Color
    expr: products.Color
  - name: ListPrice
    expr: products.ListPrice
  - name: ProductLine
    expr: products.ProductLine
  - name: ProductSubcategoryID
    expr: products.ProductSubcategoryID
measures:
  - name: total_quantity
    expr: SUM(source.OrderQty)
$$
```

In [ ]:
SCHEMAS = dict.fromkeys(QUESTIONS, schema)

metric_view = dbutils.widgets.get('metric_view')
if metric_view:
    view_schema = Schema.from_source(source.semantic_model(metric_view))
    for view_table in view_schema.tables.values():
        print(view_table)
    print(view_schema.relationships)

    # The view's source table keeps the name it has in the catalog.
    lines = 'adventureworks_sales_order_details'
    QUESTIONS['view_demand_30d'] = (
        f'PREDICT SUM({lines}.OrderQty, 0, 30, days) FOR EACH products.ProductID',
        {'indices': [top_product], 'inference_config': ENSEMBLE},
    )
    QUESTIONS['view_inactivity_90d'] = (
        f'PREDICT COUNT({lines}.*, 0, 90, days)=0 FOR EACH customers.CustomerID',
        {'indices': frequent_customers},
    )
    SCHEMAS['view_demand_30d'] = view_schema
    SCHEMAS['view_inactivity_90d'] = view_schema

In [ ]:
from sdm_client import (
    KumoRelational,
    PredictiveQuery,
    SDMClient,
    StructuredDataError,
)

# A question the data cannot answer is refused with the reason, and the rest
# are still asked.
results, refused = {}, {}
with SDMClient.for_databricks_serving(endpoint) as client:
    for name, (text, options) in QUESTIONS.items():
        try:
            request = PredictiveQuery(text, SCHEMAS[name]).materialize(
                **options
            )
            answer = client.predict(request, model=KumoRelational())
        except StructuredDataError as error:
            refused[name] = str(error)
            print(f'{name}: refused – {error}')
            continue
        results[name] = answer.to_pandas()
        print(f'{name}: {len(results[name])} rows')
        display(results[name])

In [ ]:
dbutils.notebook.exit(
    json.dumps(
        {
            'environment': ENVIRONMENT,
            'results': {
                name: frame.to_dict('records')
                for name, frame in results.items()
            },
            'refused': refused,
        },
        default=str,
    )
)